# Business Impact & SHAP Interpretability

## Supply Chain Late Delivery Prediction

---

### Purpose

This notebook translates ML model performance into **actionable business insights** through:

1. **Cost-Benefit Analysis**: Quantify ROI of the prediction system
2. **SHAP Explanations**: Understand what drives predictions
3. **Interactive Exploration**: Allow stakeholders to explore insights
4. **Business Recommendations**: Actionable strategies based on model insights

### Key Questions Answered

| Question | Analysis |
|----------|----------|
| How much money can we save? | ROI calculation |
| What features drive late deliveries? | SHAP feature importance |
| Why is this specific order flagged? | Local SHAP explanations |
| What segments are highest risk? | Segment analysis |
| What actions should we take? | Business recommendations |

---

In [ ]:
# ============================================================
# SETUP
# ============================================================
import sys
import warnings
warnings.filterwarnings('ignore')
sys.path.append('..')

import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "plotly_white"

# Interactive widgets
try:
    import ipywidgets as widgets
    from IPython.display import display, HTML
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("ipywidgets not available - interactive features disabled")

# SHAP
import shap
shap.initjs()

# Matplotlib for SHAP plots
import matplotlib.pyplot as plt
%matplotlib inline

print("Libraries loaded successfully")

In [ ]:
# ============================================================
# LOAD DATA AND MODEL
# ============================================================
from src.data.preprocess import load_or_preprocess
from src.features.build_features import build_features_pipeline

# Load data
df = load_or_preprocess()
X, y = build_features_pipeline(df)

# Split data (same as training)
RANDOM_STATE = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Load best model
model_dir = Path('../models')
model_files = sorted(model_dir.glob('best_model*.pkl'))

if model_files:
    best_model = joblib.load(model_files[-1])
    print(f"Model loaded: {model_files[-1].name}")
    
    # Align features with model if needed
    if hasattr(best_model, 'feature_names_in_'):
        model_features = list(best_model.feature_names_in_)
        X_test = X_test[model_features]
        X_train = X_train[model_features]
        print(f"Aligned features: {len(model_features)} features")
else:
    print("No model found - training quick model for demo")
    from sklearn.ensemble import RandomForestClassifier
    best_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    best_model.fit(X_train, y_train)

# Generate predictions
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print(f"\nTest set: {len(y_test):,} orders")
print(f"Model accuracy: {(y_pred == y_test).mean():.1%}")

---

## 1. Business Cost Analysis

### Cost Assumptions

| Cost Type | Amount | Description |
|-----------|--------|-------------|
| **Late Delivery Cost** | $75 | Customer service, refunds, reputation damage |
| **Intervention Cost** | $15 | Shipping upgrade, proactive communication |
| **Revenue Saved** | $50 | Revenue retained by preventing late delivery |

These assumptions can be adjusted based on your business context.

In [ ]:
# ============================================================
# COST-BENEFIT ANALYSIS
# ============================================================

# Business cost assumptions
COST_PER_LATE_DELIVERY = 75
COST_PER_INTERVENTION = 15
REVENUE_SAVED_PER_CATCH = 50

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

# Scale to annual (full dataset)
scale_factor = len(y) / len(y_test)
total_orders = len(y)
late_orders = y.sum()

# SCENARIO 1: No ML (Current State)
scenario1_cost = late_orders * COST_PER_LATE_DELIVERY

# SCENARIO 2: With ML Model
caught_late = tp * scale_factor
missed_late = fn * scale_factor
false_alarms = fp * scale_factor

scenario2_late_cost = missed_late * COST_PER_LATE_DELIVERY
scenario2_intervention_cost = (caught_late + false_alarms) * COST_PER_INTERVENTION
scenario2_revenue_saved = caught_late * REVENUE_SAVED_PER_CATCH
scenario2_cost = scenario2_late_cost + scenario2_intervention_cost - scenario2_revenue_saved

# Net savings
net_savings = scenario1_cost - scenario2_cost
roi_pct = (net_savings / scenario1_cost) * 100 if scenario1_cost > 0 else 0

print("COST-BENEFIT ANALYSIS")
print("=" * 60)
print(f"\nSCENARIO 1: Without ML")
print(f"   Late delivery costs: ${scenario1_cost:,.0f}")
print(f"\nSCENARIO 2: With ML")
print(f"   Late deliveries caught: {caught_late:,.0f}")
print(f"   Late deliveries missed: {missed_late:,.0f}")
print(f"   False alarms: {false_alarms:,.0f}")
print(f"   Total cost: ${scenario2_cost:,.0f}")
print(f"\nNET ANNUAL SAVINGS: ${net_savings:,.0f} ({roi_pct:.1f}% reduction)")

In [ ]:
# Visualize cost analysis
fig = make_subplots(
    rows=1, cols=3,
    specs=[[{"type": "bar"}, {"type": "indicator"}, {"type": "pie"}]],
    subplot_titles=('<b>Cost Comparison</b>', '<b>Annual Savings</b>', '<b>Model Predictions</b>')
)

# Cost comparison
fig.add_trace(
    go.Bar(
        x=['Without ML', 'With ML'],
        y=[scenario1_cost, max(0, scenario2_cost)],
        marker_color=['#e74c3c', '#2ecc71'],
        text=[f'${scenario1_cost:,.0f}', f'${max(0, scenario2_cost):,.0f}'],
        textposition='outside'
    ),
    row=1, col=1
)

# Savings indicator
fig.add_trace(
    go.Indicator(
        mode="number",
        value=net_savings,
        number={'prefix': '$', 'valueformat': ',.0f', 'font': {'size': 40, 'color': '#27ae60'}},
        title={'text': f'{roi_pct:.0f}% Reduction'}
    ),
    row=1, col=2
)

# Prediction breakdown
fig.add_trace(
    go.Pie(
        labels=['True Positives', 'True Negatives', 'False Positives', 'False Negatives'],
        values=[tp, tn, fp, fn],
        marker_colors=['#27ae60', '#3498db', '#f39c12', '#e74c3c'],
        textinfo='percent+label',
        hole=0.3
    ),
    row=1, col=3
)

fig.update_layout(height=400, title='<b>Business Impact Dashboard</b>', showlegend=False)
fig.show()

---

## 2. SHAP Model Interpretability

### What is SHAP?

**SHAP (SHapley Additive exPlanations)** is a game-theoretic approach to explain predictions:

- **Positive SHAP value**: Feature pushes prediction toward "Late"
- **Negative SHAP value**: Feature pushes prediction toward "On-Time"
- **Magnitude**: How much the feature contributes

### Business Value of SHAP

| Use Case | How SHAP Helps |
|----------|----------------|
| **Explain predictions** | Show stakeholders why an order is flagged |
| **Identify risk factors** | Understand what drives late deliveries |
| **Validate model** | Ensure predictions align with domain knowledge |
| **Guide interventions** | Target specific factors for improvement |

In [ ]:
# ============================================================
# SHAP EXPLAINER INITIALIZATION
# ============================================================

print("Initializing SHAP explainer...")

# Use appropriate explainer based on model type
model_type = type(best_model).__name__
print(f"Model type: {model_type}")

# Sample for faster computation
SHAP_SAMPLE_SIZE = 500
sample_idx = np.random.choice(len(X_test), min(SHAP_SAMPLE_SIZE, len(X_test)), replace=False)
X_shap = X_test.iloc[sample_idx].copy()
y_shap = y_test.iloc[sample_idx].copy()
y_proba_shap = y_proba[sample_idx]

# Create explainer
if 'LGBM' in model_type or 'XGB' in model_type or 'CatBoost' in model_type:
    explainer = shap.TreeExplainer(best_model)
    explainer_type = "TreeExplainer"
elif 'RandomForest' in model_type or 'GradientBoosting' in model_type:
    explainer = shap.TreeExplainer(best_model)
    explainer_type = "TreeExplainer"
elif 'LogisticRegression' in model_type:
    explainer = shap.LinearExplainer(best_model, X_train)
    explainer_type = "LinearExplainer"
else:
    # Fallback to KernelExplainer
    background = shap.sample(X_train, 100)
    explainer = shap.KernelExplainer(best_model.predict_proba, background)
    explainer_type = "KernelExplainer"

print(f"Using {explainer_type}")

# Calculate SHAP values
print("\nCalculating SHAP values (this may take a moment)...")
shap_values = explainer.shap_values(X_shap)

# Handle binary classification output
if isinstance(shap_values, list):
    shap_values = shap_values[1]  # Positive class (Late)

print(f"\nSHAP values calculated for {len(X_shap)} samples")
print(f"Shape: {shap_values.shape}")

In [ ]:
# ============================================================
# SHAP SUMMARY PLOT (Global Feature Importance)
# ============================================================

print("GLOBAL FEATURE IMPORTANCE (SHAP)")
print("=" * 60)
print("\nThis plot shows which features have the biggest impact on predictions:")
print("- Features at the top are most important")
print("- Red = high feature value, Blue = low feature value")
print("- Right side = pushes prediction toward 'Late'")
print("- Left side = pushes prediction toward 'On-Time'")

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_shap, show=False, max_display=15)
plt.title('SHAP Feature Importance - What Drives Late Delivery Predictions?', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SHAP BAR PLOT (Average Absolute Importance)
# ============================================================

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_shap, plot_type='bar', show=False, max_display=15)
plt.title('Average SHAP Value Magnitude by Feature', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

# Create feature importance dataframe
mean_abs_shap = np.abs(shap_values).mean(axis=0)
feature_importance_df = pd.DataFrame({
    'feature': X_shap.columns,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False)

print("\nTop 10 Features by SHAP Importance:")
print(feature_importance_df.head(10).to_string(index=False))

In [ ]:
# ============================================================
# SHAP DEPENDENCE PLOTS (Top Features)
# ============================================================

print("FEATURE DEPENDENCE PLOTS")
print("=" * 60)
print("\nThese plots show how each feature's value affects predictions:")

top_features = feature_importance_df['feature'].head(4).tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feature in enumerate(top_features):
    if feature in X_shap.columns:
        ax = axes[i]
        shap.dependence_plot(
            feature, 
            shap_values, 
            X_shap,
            interaction_index=None,
            ax=ax,
            show=False
        )
        ax.set_title(f'{feature}', fontsize=12)

plt.suptitle('Feature Dependence: How Feature Values Affect Predictions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## 3. Interactive SHAP Exploration

### Local Explanations

Understand why individual orders are predicted as late or on-time.

In [ ]:
# ============================================================
# INTERACTIVE LOCAL EXPLANATIONS
# ============================================================

# Get base value
if isinstance(explainer.expected_value, np.ndarray):
    base_value = explainer.expected_value[1]
else:
    base_value = explainer.expected_value

def explain_order(order_idx):
    """Generate SHAP explanation for a specific order."""
    order_data = X_shap.iloc[order_idx]
    order_shap = shap_values[order_idx]
    actual = y_shap.iloc[order_idx]
    predicted_proba = y_proba_shap[order_idx]
    
    print(f"\nORDER EXPLANATION (Index: {order_idx})")
    print("=" * 60)
    print(f"Predicted probability of late delivery: {predicted_proba:.1%}")
    print(f"Actual outcome: {'LATE' if actual == 1 else 'ON-TIME'}")
    print(f"Prediction: {'LATE' if predicted_proba >= 0.5 else 'ON-TIME'}")
    print(f"{'CORRECT' if (predicted_proba >= 0.5) == actual else 'INCORRECT'}")
    
    # Top contributing features
    feature_contributions = pd.DataFrame({
        'feature': X_shap.columns,
        'value': order_data.values,
        'shap_value': order_shap
    }).sort_values('shap_value', key=abs, ascending=False)
    
    print("\nTop 5 Contributing Factors:")
    for _, row in feature_contributions.head(5).iterrows():
        direction = 'Late' if row['shap_value'] > 0 else 'On-Time'
        print(f"   {row['feature']}: {row['value']:.2f}")
        print(f"      SHAP: {row['shap_value']:+.4f} (pushes toward {direction})")
    
    return order_data, order_shap

# Example: Explain a high-risk order
high_risk_idx = np.argmax(y_proba_shap)
print("\nHIGH-RISK ORDER EXAMPLE")
order_data, order_shap = explain_order(high_risk_idx)

# Waterfall plot
plt.figure(figsize=(12, 6))
shap.waterfall_plot(
    shap.Explanation(
        values=order_shap,
        base_values=base_value,
        data=order_data.values,
        feature_names=X_shap.columns.tolist()
    ),
    max_display=10,
    show=False
)
plt.title(f'SHAP Waterfall: High-Risk Order (Prob={y_proba_shap[high_risk_idx]:.1%})', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Example: Explain a low-risk order
low_risk_idx = np.argmin(y_proba_shap)
print("\nLOW-RISK ORDER EXAMPLE")
order_data, order_shap = explain_order(low_risk_idx)

# Waterfall plot
plt.figure(figsize=(12, 6))
shap.waterfall_plot(
    shap.Explanation(
        values=order_shap,
        base_values=base_value,
        data=order_data.values,
        feature_names=X_shap.columns.tolist()
    ),
    max_display=10,
    show=False
)
plt.title(f'SHAP Waterfall: Low-Risk Order (Prob={y_proba_shap[low_risk_idx]:.1%})', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# INTERACTIVE ORDER EXPLORER
# ============================================================

if WIDGETS_AVAILABLE:
    print("INTERACTIVE ORDER EXPLORER")
    print("=" * 60)
    print("Use the dropdown to explore different orders")
    
    # Create dropdown options
    order_options = [
        (f"Order {i}: Prob={y_proba_shap[i]:.1%} ({'Late' if y_shap.iloc[i] == 1 else 'On-Time'})", i)
        for i in range(min(50, len(X_shap)))
    ]
    
    def update_explanation(order_idx):
        explain_order(order_idx)
        
        # Force plot (interactive)
        display(shap.force_plot(
            base_value,
            shap_values[order_idx],
            X_shap.iloc[order_idx],
            matplotlib=False
        ))
    
    order_dropdown = widgets.Dropdown(
        options=order_options,
        description='Select Order:',
        style={'description_width': 'initial'}
    )
    
    out = widgets.interactive_output(update_explanation, {'order_idx': order_dropdown})
    display(order_dropdown, out)
else:
    print("Install ipywidgets for interactive exploration: pip install ipywidgets")

---

## 4. Segment Analysis

Understand which customer segments, shipping modes, or regions have highest risk.

In [ ]:
# ============================================================
# RISK TIER ANALYSIS
# ============================================================

# Create risk tiers
risk_df = pd.DataFrame({
    'probability': y_proba,
    'actual': y_test.values,
    'predicted': y_pred
})

risk_df['risk_tier'] = pd.cut(
    risk_df['probability'],
    bins=[0, 0.3, 0.5, 0.7, 1.0],
    labels=['Low', 'Medium', 'High', 'Critical']
)

# Tier statistics
tier_stats = risk_df.groupby('risk_tier').agg({
    'probability': 'count',
    'actual': 'mean'
}).rename(columns={'probability': 'count', 'actual': 'actual_late_rate'})
tier_stats['actual_late_rate'] = tier_stats['actual_late_rate'] * 100

print("RISK TIER ANALYSIS")
print("=" * 60)
print(tier_stats.to_string())

# Visualization
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>Orders by Risk Tier</b>', '<b>Actual Late Rate by Tier</b>')
)

tier_colors = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e67e22', 'Critical': '#e74c3c'}

fig.add_trace(
    go.Bar(
        x=tier_stats.index.tolist(),
        y=tier_stats['count'].tolist(),
        marker_color=[tier_colors[t] for t in tier_stats.index],
        text=[f"{v:,}" for v in tier_stats['count']],
        textposition='outside'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Bar(
        x=tier_stats.index.tolist(),
        y=tier_stats['actual_late_rate'].tolist(),
        marker_color=[tier_colors[t] for t in tier_stats.index],
        text=[f"{v:.1f}%" for v in tier_stats['actual_late_rate']],
        textposition='outside'
    ),
    row=1, col=2
)

fig.update_layout(height=400, title='<b>Risk Tier Distribution</b>', showlegend=False)
fig.show()

In [ ]:
# ============================================================
# INTERVENTION STRATEGY
# ============================================================

print("\nRECOMMENDED INTERVENTION STRATEGY")
print("=" * 70)

interventions = {
    'Low': 'Standard processing - no action needed',
    'Medium': 'Monitor shipment - send tracking to customer',
    'High': 'Proactive notification - alert customer of potential delay',
    'Critical': 'Upgrade shipping + notify customer + alert operations team'
}

for tier, action in interventions.items():
    if tier in tier_stats.index:
        count = tier_stats.loc[tier, 'count']
        rate = tier_stats.loc[tier, 'actual_late_rate']
        print(f"\n{tier.upper()} RISK ({count:,} orders, {rate:.1f}% late rate):")
        print(f"   Action: {action}")

---

## 5. Business Insights & Recommendations

### Translating SHAP into Business Actions

In [ ]:
# ============================================================
# BUSINESS INSIGHTS FROM SHAP
# ============================================================

# Load feature descriptions
try:
    feature_desc_path = Path('../feature_description.md')
    with open(feature_desc_path, 'r') as f:
        feature_desc_content = f.read()
    FEATURE_DESC_LOADED = True
except:
    FEATURE_DESC_LOADED = False

print("BUSINESS INSIGHTS FROM MODEL")
print("=" * 80)

# Top features interpretation
top_5_features = feature_importance_df.head(5)['feature'].tolist()

feature_interpretations = {
    'scheduled_shipping_days': {
        'business_name': 'Scheduled Shipping Days',
        'insight': 'Orders with longer scheduled shipping windows have higher late delivery risk',
        'action': 'Review fulfillment SLAs for long-window shipments; consider buffer time'
    },
    'shipping_urgency': {
        'business_name': 'Shipping Urgency/Mode',
        'insight': 'Same-day and First Class shipments have lower late delivery rates',
        'action': 'Prioritize expedited shipping for high-value customers'
    },
    'shipping_mode_encoded': {
        'business_name': 'Shipping Mode',
        'insight': 'Shipping method significantly impacts delivery timing',
        'action': 'Analyze carrier performance by mode; negotiate SLAs'
    },
    'customer_order_count': {
        'business_name': 'Customer Order History',
        'insight': 'Repeat customers may have different delivery patterns',
        'action': 'Segment customers by order frequency for targeted interventions'
    },
    'order_value': {
        'business_name': 'Order Value',
        'insight': 'High-value orders may warrant expedited handling',
        'action': 'Implement value-based prioritization in fulfillment'
    },
    'is_weekend': {
        'business_name': 'Weekend Orders',
        'insight': 'Weekend orders may experience processing delays',
        'action': 'Set customer expectations for weekend order processing'
    },
    'profit_margin_pct': {
        'business_name': 'Profit Margin',
        'insight': 'Order profitability correlates with fulfillment priority',
        'action': 'Balance margin optimization with customer satisfaction'
    }
}

print("\nKEY FEATURE INSIGHTS:")
print("-" * 80)

for feature in top_5_features:
    # Try to match feature to interpretation
    feature_lower = feature.lower()
    matched = False
    
    for key, interp in feature_interpretations.items():
        if key in feature_lower or feature_lower in key:
            importance = feature_importance_df[feature_importance_df['feature'] == feature]['mean_abs_shap'].values[0]
            print(f"\n{interp['business_name']} (Importance: {importance:.4f})")
            print(f"   Insight: {interp['insight']}")
            print(f"   Action:  {interp['action']}")
            matched = True
            break
    
    if not matched:
        importance = feature_importance_df[feature_importance_df['feature'] == feature]['mean_abs_shap'].values[0]
        print(f"\n{feature} (Importance: {importance:.4f})")
        print(f"   Insight: This feature significantly impacts late delivery predictions")
        print(f"   Action:  Investigate relationship with domain experts")

In [ ]:
# ============================================================
# EXECUTIVE SUMMARY DASHBOARD
# ============================================================

late_rate = y.mean() * 100
recall = tp / (tp + fn) * 100 if (tp + fn) > 0 else 0
accuracy = (tp + tn) / (tp + tn + fp + fn) * 100

fig = make_subplots(
    rows=2, cols=3,
    specs=[[{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
           [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}]],
    subplot_titles=(
        'Total Orders', 'Late Delivery Rate', 'Model Accuracy',
        'Late Deliveries Caught', 'Annual Savings', 'ROI'
    )
)

# Row 1
fig.add_trace(go.Indicator(
    mode="number", value=total_orders,
    number={'valueformat': ',', 'font': {'size': 36, 'color': '#3498db'}}
), row=1, col=1)

fig.add_trace(go.Indicator(
    mode="number", value=late_rate,
    number={'suffix': '%', 'font': {'size': 36, 'color': '#e74c3c'}}
), row=1, col=2)

fig.add_trace(go.Indicator(
    mode="number", value=accuracy,
    number={'suffix': '%', 'font': {'size': 36, 'color': '#2ecc71'}}
), row=1, col=3)

# Row 2
fig.add_trace(go.Indicator(
    mode="number", value=recall,
    number={'suffix': '%', 'font': {'size': 36, 'color': '#9b59b6'}}
), row=2, col=1)

fig.add_trace(go.Indicator(
    mode="number", value=net_savings,
    number={'prefix': '$', 'valueformat': ',.0f', 'font': {'size': 36, 'color': '#27ae60'}}
), row=2, col=2)

fig.add_trace(go.Indicator(
    mode="number", value=roi_pct,
    number={'suffix': '%', 'font': {'size': 36, 'color': '#f39c12'}}
), row=2, col=3)

fig.update_layout(
    height=450,
    title='<b>Executive Summary Dashboard</b>',
    paper_bgcolor='#f8f9fa'
)
fig.show()

In [ ]:
# ============================================================
# FINAL RECOMMENDATIONS
# ============================================================

print("\n" + "=" * 80)
print("EXECUTIVE SUMMARY & RECOMMENDATIONS")
print("=" * 80)

print(f"""
KEY FINDINGS
{"="*60}

1. PROBLEM SCOPE
   - {late_rate:.1f}% of orders experience late delivery
   - Estimated annual cost without ML: ${scenario1_cost:,.0f}

2. MODEL PERFORMANCE
   - {recall:.0f}% of late deliveries predicted before they happen
   - {accuracy:.0f}% overall prediction accuracy
   - False alarm rate: {fp/(fp+tn)*100:.1f}%

3. BUSINESS IMPACT
   - Estimated annual savings: ${net_savings:,.0f}
   - ROI: {roi_pct:.0f}% cost reduction
   - Break-even: Immediate

4. KEY DRIVERS (from SHAP)
   - Top factors: {', '.join(top_5_features[:3])}
   - These features should be prioritized for operational improvements

RECOMMENDATIONS
{"="*60}

1. IMMEDIATE (0-30 days)
   - Deploy model in production for real-time scoring
   - Implement risk-tier based intervention workflow
   - Train customer service team on model outputs

2. SHORT-TERM (1-3 months)
   - A/B test intervention strategies by risk tier
   - Monitor model performance and recalibrate thresholds
   - Build dashboard for operations team

3. LONG-TERM (3-6 months)
   - Automate shipping upgrades for critical-risk orders
   - Integrate with carrier APIs for real-time updates
   - Retrain model monthly with new data

NEXT STEPS
{"="*60}

1. Review this analysis with stakeholders
2. Validate cost assumptions with finance team
3. Plan pilot deployment on 10% of orders
4. Define success metrics for pilot
""")